# 01 tmdb id collection

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

print("Project folder exists:", os.path.exists(PROJECT_ROOT))
print("Project path:", PROJECT_ROOT)

In [ ]:
for folder in [
    "01_data/raw",
    "01_data/processed",
    "01_data/final",
    "02_notebooks",
    "03_scripts",
    "04_config",
    "05_logs",
    "06_outputs/tmdb",
    "06_outputs/subtitles",
    "06_outputs/reviews",
    "06_outputs/nlp",
    "06_outputs/visualizations",
    "07_documentation"
]:
    path = os.path.join(PROJECT_ROOT, folder)
    print("✓", folder, "→", os.path.exists(path))

In [ ]:
import pandas as pd
import os

MASTER_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_MASTER_DATA.csv"
)

df = pd.read_csv(MASTER_DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

In [ ]:
df.head()

In [ ]:
print("Years:", df["year"].min(), "to", df["year"].max())
print("Total movies:", len(df))
print("Movies per year:")
print(df.groupby("year").size())

In [ ]:
df[[
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "india_gross_cr"
]].head(10)

In [ ]:
from google.colab import userdata

TMDB_API_KEY = userdata.get("TMDB_API_KEY")

if TMDB_API_KEY:
    print("TMDB API key loaded successfully.")
else:
    print("TMDB API key was not found.")

In [ ]:
import requests

url = "https://api.themoviedb.org/3/search/movie"

params = {
    "api_key": TMDB_API_KEY,
    "query": "Shivaji",
    "year": 2007
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("Response received:", response.ok)

In [ ]:
data = response.json()

print("Number of results:", len(data.get("results", [])))

for movie in data.get("results", []):
    print(
        movie.get("id"),
        "|",
        movie.get("title"),
        "|",
        movie.get("release_date"),
        "|",
        movie.get("original_language")
    )

In [ ]:
def search_tmdb_movie(title, year):
    url = "https://api.themoviedb.org/3/search/movie"

    params = {
        "api_key": TMDB_API_KEY,
        "query": title,
        "year": int(year),
        "include_adult": False
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    return data.get("results", [])

In [ ]:
results = search_tmdb_movie("Shivaji", 2007)

for movie in results:
    print(
        "TMDB ID:", movie.get("id"),
        "| Title:", movie.get("title"),
        "| Release:", movie.get("release_date"),
        "| Language:", movie.get("original_language")
    )

Industry Mapping

In [ ]:
INDUSTRY_MAP = {
    "hindi": "Bollywood",
    "telugu": "Tollywood",
    "tamil": "Kollywood",
    "malayalam": "Mollywood",
    "kannada": "Sandalwood",
    "bengali": "Bengali Cinema",
    "marathi": "Marathi Cinema",
    "punjabi": "Punjabi Cinema",
    "gujarati": "Gujarati Cinema",
    "assamese": "Assamese Cinema",
    "odia": "Odia Cinema",
}

In [ ]:
def classify_industry(language):
    language = str(language).strip().lower()

    return INDUSTRY_MAP.get(
        language,
        "Other Indian Cinema"
    )

df["industry"] = df["language"].apply(classify_industry)

df[[
    "movie_record_id",
    "year",
    "movie_title",
    "language",
    "industry"
]].head(20)

Create the mapping dataframe

In [ ]:
tmdb_mapping = df[[
    "movie_record_id",
    "year",
    "rank",
    "movie_title",
    "language",
    "industry"
]].copy()

tmdb_mapping["tmdb_id"] = pd.NA
tmdb_mapping["tmdb_title"] = pd.NA
tmdb_mapping["tmdb_release_date"] = pd.NA
tmdb_mapping["tmdb_original_language"] = pd.NA
tmdb_mapping["tmdb_match_status"] = "Pending"
tmdb_mapping["tmdb_match_notes"] = pd.NA

print("TMDB mapping table created.")
print("Rows:", len(tmdb_mapping))
print("Columns:", len(tmdb_mapping.columns))

In [ ]:
tmdb_mapping.head()

In [ ]:
import time

REQUEST_DELAY = 0.25

In [ ]:
tmdb_candidates = []

for _, row in tmdb_mapping.iterrows():

    title = row["movie_title"]
    year = row["year"]

    try:
        results = search_tmdb_movie(title, year)

        tmdb_candidates.append({
            "movie_record_id": row["movie_record_id"],
            "year": year,
            "rank": row["rank"],
            "movie_title": title,
            "language": row["language"],
            "industry": row["industry"],
            "candidate_count": len(results),
            "candidates": results
        })

        print(
            row["movie_record_id"],
            "|",
            title,
            "| candidates:",
            len(results)
        )

    except Exception as e:

        tmdb_candidates.append({
            "movie_record_id": row["movie_record_id"],
            "year": year,
            "rank": row["rank"],
            "movie_title": title,
            "language": row["language"],
            "industry": row["industry"],
            "candidate_count": 0,
            "candidates": [],
            "error": str(e)
        })

        print(
            row["movie_record_id"],
            "|",
            title,
            "| ERROR:",
            e
        )

    time.sleep(REQUEST_DELAY)

print("\nTMDB candidate search completed.")
print("Movies processed:", len(tmdb_candidates))

In [ ]:
print("Total movies processed:", len(tmdb_candidates))

print("\nMovies with zero candidates:")

for item in tmdb_candidates:
    if item["candidate_count"] == 0:
        print(
            item["movie_record_id"],
            "|",
            item["movie_title"],
            "|",
            item["year"]
        )

In [ ]:
candidate_summary = pd.DataFrame([
    {
        "movie_record_id": x["movie_record_id"],
        "movie_title": x["movie_title"],
        "year": x["year"],
        "candidate_count": x["candidate_count"]
    }
    for x in tmdb_candidates
])

candidate_summary["candidate_count"].value_counts().sort_index()

In [ ]:
import json
import os

TMDB_CANDIDATES_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_TMDB_CANDIDATES.json"
)

with open(TMDB_CANDIDATES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        tmdb_candidates,
        f,
        ensure_ascii=False,
        indent=2
    )

print("TMDB candidate results saved successfully.")
print(TMDB_CANDIDATES_PATH)

In [ ]:
for item in tmdb_candidates:
    if item["candidate_count"] == 0:
        print("Movie:", item["movie_title"])
        print("Year:", item["year"])
        print("Language:", item["language"])
        print("Industry:", item["industry"])

In [ ]:
test_titles = [
    "Lage Raho Munna Bhai",
    "Lage Raho MunnaBhai",
    "Lage Raho Munnabhai"
]

for title in test_titles:
    results = search_tmdb_movie(title, 2006)

    print("\nSEARCH:", title)
    print("Results:", len(results))

    for movie in results[:5]:
        print(
            movie.get("id"),
            "|",
            movie.get("title"),
            "|",
            movie.get("release_date"),
            "|",
            movie.get("original_language")
        )

Build the title normalization function

In [ ]:
import re
import unicodedata

def normalize_title(title):
    if pd.isna(title):
        return ""

    title = str(title)

    # Normalize Unicode characters
    title = unicodedata.normalize("NFKD", title)

    # Convert to lowercase
    title = title.lower()

    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Normalize whitespace
    title = re.sub(r"\s+", " ", title).strip()

    return title

In [ ]:
test_titles = [
    "Lage Raho Munnabhai",
    "Lage Raho Munna Bhai",
    "Sivaji: The Boss",
    "Shivaji"
]

for title in test_titles:
    print(title, "→", normalize_title(title))

Install/import fuzzy matching

In [ ]:
from difflib import SequenceMatcher

In [ ]:
def title_similarity(title1, title2):
    t1 = normalize_title(title1)
    t2 = normalize_title(title2)

    if not t1 or not t2:
        return 0.0

    # Different fuzzy comparison methods
    ratio_score = fuzz.ratio(t1, t2) / 100
    partial_score = fuzz.partial_ratio(t1, t2) / 100
    token_score = fuzz.token_set_ratio(t1, t2) / 100

    # Use the strongest evidence
    return max(
        ratio_score,
        partial_score,
        token_score
    )

In [ ]:
tests = [
    ("Shivaji", "Sivaji: The Boss"),
    ("Lage Raho Munnabhai", "Lage Raho Munna Bhai"),
    ("Kaho Naa... Pyaar Hai", "Kaho Naa... Pyaar Hai"),
]

for a, b in tests:
    print(
        a,
        "vs",
        b,
        "→",
        round(title_similarity(a, b), 3)
    )

Create language-code mapping

In [ ]:
LANGUAGE_CODE_MAP = {
    "hindi": "hi",
    "telugu": "te",
    "tamil": "ta",
    "malayalam": "ml",
    "kannada": "kn",
    "bengali": "bn",
    "marathi": "mr",
    "punjabi": "pa",
    "gujarati": "gu",
    "assamese": "as",
    "odia": "or",
}

In [ ]:
def language_matches(dataset_language, tmdb_language):
    expected = LANGUAGE_CODE_MAP.get(
        str(dataset_language).strip().lower()
    )

    return expected == tmdb_language

In [ ]:
print(language_matches("Hindi", "hi"))
print(language_matches("Telugu", "te"))
print(language_matches("Hindi", "ta"))

Create the candidate scoring function

In [ ]:
def score_tmdb_candidate(
    dataset_title,
    dataset_year,
    dataset_language,
    candidate
):
    tmdb_title = candidate.get("title", "")
    release_date = candidate.get("release_date", "")
    tmdb_language = candidate.get("original_language", "")

    # Title similarity
    title_score = title_similarity(
        dataset_title,
        tmdb_title
    )

    # Year match
    year_score = 0

    if release_date:
        try:
            tmdb_year = int(release_date[:4])

            if tmdb_year == int(dataset_year):
                year_score = 1
            elif abs(tmdb_year - int(dataset_year)) == 1:
                year_score = 0.5

        except (ValueError, TypeError):
            pass

    # Language match
    language_score = (
        1 if language_matches(
            dataset_language,
            tmdb_language
        )
        else 0
    )

    # Weighted score
    final_score = (
        0.60 * title_score +
        0.25 * year_score +
        0.15 * language_score
    )

    return {
        "tmdb_id": candidate.get("id"),
        "tmdb_title": tmdb_title,
        "tmdb_release_date": release_date,
        "tmdb_original_language": tmdb_language,
        "title_score": round(title_score, 4),
        "year_score": year_score,
        "language_score": language_score,
        "final_score": round(final_score, 4)
    }

In [ ]:
shivaji_results = search_tmdb_movie("Shivaji", 2007)

for candidate in shivaji_results:
    scored = score_tmdb_candidate(
        "Shivaji",
        2007,
        "Tamil",
        candidate
    )

    print(scored)

In [ ]:
Lage_Raho_results = search_tmdb_movie("Lage Raho Munna Bhai", 2006)

for candidate in Lage_Raho_results:
    scored = score_tmdb_candidate(
        "Lage Raho Munna Bhai",
        2006,
        "Hindi",
        candidate
    )

    print(scored)

In [ ]:
all_scored_candidates = []

for item in tmdb_candidates:

    dataset_title = item["movie_title"]
    dataset_year = item["year"]
    dataset_language = item["language"]

    for candidate in item["candidates"]:

        scored = score_tmdb_candidate(
            dataset_title,
            dataset_year,
            dataset_language,
            candidate
        )

        scored["movie_record_id"] = item["movie_record_id"]
        scored["dataset_title"] = dataset_title
        scored["dataset_year"] = dataset_year
        scored["dataset_language"] = dataset_language
        scored["industry"] = item["industry"]
        scored["candidate_count"] = item["candidate_count"]

        all_scored_candidates.append(scored)

scored_candidates_df = pd.DataFrame(all_scored_candidates)

print("Total candidate records:", len(scored_candidates_df))
print("Movies represented:", scored_candidates_df["movie_record_id"].nunique())

In [ ]:
best_candidates = (
    scored_candidates_df
    .sort_values(
        ["movie_record_id", "final_score"],
        ascending=[True, False]
    )
    .groupby("movie_record_id")
    .head(1)
    .reset_index(drop=True)
)

print("Best candidates generated:", len(best_candidates))

Show the lowest-scoring matches

In [ ]:
review_candidates = best_candidates[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "dataset_language",
        "tmdb_id",
        "tmdb_title",
        "tmdb_release_date",
        "tmdb_original_language",
        "title_score",
        "year_score",
        "language_score",
        "final_score",
        "candidate_count"
    ]
].sort_values("final_score")

review_candidates.head(20)

Also find movies with many candidates

In [ ]:
ambiguous_candidates = best_candidates[
    best_candidates["candidate_count"] >= 5
].sort_values(
    ["candidate_count", "final_score"],
    ascending=[False, True]
)

ambiguous_candidates[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "dataset_language",
        "tmdb_id",
        "tmdb_title",
        "tmdb_release_date",
        "tmdb_original_language",
        "title_score",
        "final_score",
        "candidate_count"
    ]
]

In [ ]:
!pip -q install rapidfuzz

In [ ]:
from rapidfuzz import fuzz

In [ ]:
tests = [
    ("Lagaan", "Lagaan: Once Upon a Time in India"),
    ("Enthiran (Robot)", "Enthiran"),
    ("Ponniyin Selvan: I", "Ponniyin Selvan: Part I"),
    ("Shivaji", "Sivaji: The Boss"),
    ("Lage Raho Munnabhai", "Lage Raho Munna Bhai"),
    ("Kaho Naa... Pyaar Hai", "Kaho Naa... Pyaar Hai")
]

for a, b in tests:
    print(
        a,
        "vs",
        b,
        "→",
        round(title_similarity(a, b), 4)
    )

In [ ]:
best_candidates[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "dataset_language",
        "tmdb_id",
        "tmdb_title",
        "tmdb_release_date",
        "tmdb_original_language",
        "title_score",
        "year_score",
        "language_score",
        "final_score",
        "candidate_count"
    ]
].sort_values("final_score").head(20)

Check the gap between 1st and 2nd candidates

In [ ]:
candidate_ranked = (
    scored_candidates_df
    .sort_values(
        ["movie_record_id", "final_score"],
        ascending=[True, False]
    )
)

top_two = (
    candidate_ranked
    .groupby("movie_record_id")
    .head(2)
)

candidate_comparison = (
    top_two
    .groupby("movie_record_id")
    .agg(
        first_score=("final_score", "first"),
        second_score=("final_score", lambda x: x.iloc[1] if len(x) > 1 else None),
        first_tmdb_id=("tmdb_id", "first"),
        first_tmdb_title=("tmdb_title", "first"),
        second_tmdb_id=("tmdb_id", lambda x: x.iloc[1] if len(x) > 1 else None),
        second_tmdb_title=("tmdb_title", lambda x: x.iloc[1] if len(x) > 1 else None),
        candidate_count=("candidate_count", "first")
    )
    .reset_index()
)

candidate_comparison["score_gap"] = (
    candidate_comparison["first_score"] -
    candidate_comparison["second_score"].fillna(0)
)

candidate_comparison.sort_values("score_gap").head(20)

Find all candidates that need attention

In [ ]:
manual_review = best_candidates[
    (best_candidates["final_score"] < 0.90) |
    (best_candidates["title_score"] < 0.80)
].copy()

manual_review = manual_review.sort_values(
    "final_score"
)

print("Movies requiring manual review:", len(manual_review))

manual_review[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_year",
        "dataset_language",
        "tmdb_id",
        "tmdb_title",
        "tmdb_release_date",
        "tmdb_original_language",
        "title_score",
        "year_score",
        "language_score",
        "final_score",
        "candidate_count"
    ]
]

In [ ]:
lage_raho_verified = {
    "movie_record_id": "2006_2",
    "dataset_title": "Lage Raho Munnabhai",
    "dataset_year": 2006,
    "dataset_language": "Hindi",
    "tmdb_id": 21461,
    "tmdb_title": "Lage Raho Munna Bhai",
    "tmdb_release_date": "2006-09-01",
    "tmdb_original_language": "hi",
    "match_status": "Manual_Verified"
}

print(lage_raho_verified)

In [ ]:
dataframes = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        dataframes.append((name, obj.shape))

for name, shape in dataframes:
    print(name, "→", shape)

In [ ]:
original_ids = set(df["movie_record_id"])

matched_ids = set(best_candidates["movie_record_id"])

# Add the manually verified Lage Raho Munnabhai
matched_ids.add("2006_2")

missing_ids = original_ids - matched_ids
extra_ids = matched_ids - original_ids

print("Original movie IDs:", len(original_ids))
print("Matched movie IDs:", len(matched_ids))
print("Missing IDs:", missing_ids)
print("Extra IDs:", extra_ids)

Check for duplicate TMDB IDs

In [ ]:
duplicate_tmdb_ids = (
    best_candidates
    .groupby("tmdb_id")
    .agg(
        movie_count=("movie_record_id", "count"),
        movies=("dataset_title", lambda x: list(x))
    )
    .reset_index()
)

duplicate_tmdb_ids = duplicate_tmdb_ids[
    duplicate_tmdb_ids["movie_count"] > 1
]

duplicate_tmdb_ids

Language consistency check

In [ ]:
language_check = best_candidates[
    [
        "movie_record_id",
        "dataset_title",
        "dataset_language",
        "tmdb_title",
        "tmdb_original_language",
        "final_score"
    ]
].copy()

language_check

Create the complete 130-row TMDB mapping

In [ ]:
# Start with the 129 automatically matched movies
final_tmdb_mapping = best_candidates.copy()

# Add the manually verified Lage Raho Munnabhai record
lage_raho_row = pd.DataFrame([{
    "movie_record_id": "2006_2",
    "dataset_title": "Lage Raho Munnabhai",
    "dataset_year": 2006,
    "dataset_language": "Hindi",
    "industry": "Other Indian Cinema",
    "tmdb_id": 21461,
    "tmdb_title": "Lage Raho Munna Bhai",
    "tmdb_release_date": "2006-09-01",
    "tmdb_original_language": "hi",
    "title_score": 0.9744,
    "year_score": 1.0,
    "language_score": 1.0,
    "final_score": 1.0,
    "candidate_count": 1,
    "match_status": "Manual_Verified"
}])

# Mark automatic matches
final_tmdb_mapping["match_status"] = "Auto_Matched"

# Combine
final_tmdb_mapping = pd.concat(
    [final_tmdb_mapping, lage_raho_row],
    ignore_index=True
)

# Sort by movie_record_id
final_tmdb_mapping = final_tmdb_mapping.sort_values(
    "movie_record_id"
).reset_index(drop=True)

print("Final mapping shape:", final_tmdb_mapping.shape)
print("Unique movies:", final_tmdb_mapping["movie_record_id"].nunique())

In [ ]:
print("Rows:", len(final_tmdb_mapping))
print("Unique movie_record_id:", final_tmdb_mapping["movie_record_id"].nunique())
print("Unique tmdb_id:", final_tmdb_mapping["tmdb_id"].nunique())
print("Missing TMDB IDs:", final_tmdb_mapping["tmdb_id"].isna().sum())
print("Duplicate movie IDs:", final_tmdb_mapping["movie_record_id"].duplicated().sum())
print("Duplicate TMDB IDs:", final_tmdb_mapping["tmdb_id"].duplicated().sum())

# ============================================================
# Stage 1: TMDB Movie Mapping
# Indian Cinema Trend Analytics
# ============================================================

In [ ]:
output_path = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/01_data/processed/ICMB_130_TMDB_MAPPING.csv"

final_tmdb_mapping.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully:")
print(output_path)

In [ ]:
check = pd.read_csv(output_path)

print("Rows:", len(check))
print("Columns:", len(check.columns))
print(check.head())